### <font color='teal'> ***Importing Libraries*** </font>

In [17]:
from bs4 import BeautifulSoup
import requests
import json
import random
from crewai_tools import BaseTool
from pydantic import BaseModel, Field
from typing import List, Dict, Type

In [25]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

### <font color='teal'> ***Urls to extract data from*** </font>

In [2]:
# URLS = [
#         "https://www.mylondon.news/all-about/crime",
#         "https://www.birminghammail.co.uk/all-about/crime",
#         "https://www.manchestereveningnews.co.uk/all-about/crime",
#         "https://www.liverpoolecho.co.uk/all-about/crime",
#         "https://www.walesonline.co.uk/all-about/crime",
#     ]

### <font color='teal'> ***Getting News Links*** </font>

In [3]:
# urls = []
# links = []

# link_class = ['headline']

# for URL in URLS:
#     url = URL
#     page = requests.get(url)
#     soup = BeautifulSoup(page.text, 'html')
    
#     for class_name in link_class:
#         sublist = [a.get('href') for a in soup.find_all('a', class_=class_name)]
    
#     # Select 5 random elements from the sublist
#     random_subset = random.sample(sublist, min(5, len(sublist)))
    
#     urls.append(random_subset)

# # Flatten the list of lists into a single list
# links = [link for sublist in urls for link in sublist]

### <font color='teal'> ***Setting the Classes for different elements*** </font>

In [4]:
# class_names = {
#     'title': [
#         'section-theme-background-indicator publication-font',
#         'HeadingOne_heading-one__mmNWJ __className_9cd0cd'
#     ],
#     'subtitle': [
#         'sub-title',
#         'LeadText_lead-text__wd_PA'
#     ],
#     'paragraphs': [
#         'article-body',
#         'Paragraph_paragraph-text__PVKlh'
#     ]
# }

### <font color='teal'> ***Extracting the necessary data*** </font>

In [5]:
# data = []

# for link in links:
#     try:
#         article_1 = requests.get(link)
#         article = BeautifulSoup(article_1.text, 'html.parser')

#         # Extract title
#         title = None
#         for class_name in class_names['title']:
#             title = article.find('h1', class_=class_name)
#             if title:
#                 title = title.text.strip()
#                 break

#         # Extract subtitle
#         subtitle = None
#         for class_name in class_names['subtitle']:
#             subtitle = article.find('h2', class_=class_name)
#             if subtitle:
#                 subtitle = subtitle.text.strip()
#                 break

#         # Extract paragraphs
#         paragraphs = []
#         for class_name in class_names['paragraphs']:
#             paragraphs.extend(article.find_all(['p', 'div'], class_=class_name))

#         main_paragraphs = [p.text.strip() for p in paragraphs]

#         article_data = {
#             'link': link,
#             'title': title,
#             'subtitle': subtitle,
#             'paragraphs': main_paragraphs
#         }

#         data.append(article_data)

#     except Exception as e:
#         print(f"Error occurred while scraping {link}: {str(e)}")

### <font color='teal'> ***Storing the data in json format*** </font>

In [6]:
# # Save the data to a JSON file
# with open('scraped_data.json', 'w') as f:
#     json.dump(data, f, indent=4)
#     print("Data saved to scraped_data.json")

Data saved to scraped_data.json


### <font color='orange'> ***CrewAi Custom Tool***

In [18]:
# Define the input schema for the tool
class ScrapingToolInput(BaseModel):
    urls: List[str] = Field(
        ..., 
        description="A list of URLs to scrape data from."
    )
    num_links: int = Field(
        5, 
        description="The number of random links to extract from each URL."
    )

# Define the output schema for the tool
class ScrapingToolOutput(BaseModel):
    data: List[Dict] = Field(
        ..., 
        description="A list of dictionaries containing scraped data with title, subtitle, paragraphs, and link."
    )

# Define the custom scraping tool
class WebScrapingTool(BaseTool):
    name: str = "Web Scraping Tool"
    description: str = (
        "Scrapes crime-related news articles from provided URLs. Extracts titles, subtitles, and paragraphs."
    )
    args_schema: Type[BaseModel] = ScrapingToolInput
    return_schema: Type[BaseModel] = ScrapingToolOutput

    def _run(self, urls: List[str], num_links: int = 5) -> ScrapingToolOutput:
        # Define classes for elements to scrape
        link_class = ['headline']
        class_names = {
            'title': [
                'section-theme-background-indicator publication-font',
                'HeadingOne_heading-one__mmNWJ __className_9cd0cd'
            ],
            'subtitle': [
                'sub-title',
                'LeadText_lead-text__wd_PA'
            ],
            'paragraphs': [
                'article-body',
                'Paragraph_paragraph-text__PVKlh'
            ]
        }

        links = []
        # Scrape links from provided URLs
        for url in urls:
            try:
                page = requests.get(url)
                soup = BeautifulSoup(page.text, 'html.parser')
                sublist = [a.get('href') for a in soup.find_all('a', class_=link_class[0])]
                # Select random links
                random_subset = random.sample(sublist, min(num_links, len(sublist)))
                links.extend(random_subset)
            except Exception as e:
                print(f"Error occurred while fetching links from {url}: {str(e)}")
        
        data = []
        # Scrape data from links
        for link in links:
            try:
                article_1 = requests.get(link)
                article = BeautifulSoup(article_1.text, 'html.parser')

                # Extract title
                title = None
                for class_name in class_names['title']:
                    title = article.find('h1', class_=class_name)
                    if title:
                        title = title.text.strip()
                        break

                # Extract subtitle
                subtitle = None
                for class_name in class_names['subtitle']:
                    subtitle = article.find('h2', class_=class_name)
                    if subtitle:
                        subtitle = subtitle.text.strip()
                        break

                # Extract paragraphs
                paragraphs = []
                for class_name in class_names['paragraphs']:
                    paragraphs.extend(article.find_all(['p', 'div'], class_=class_name))

                main_paragraphs = [p.text.strip() for p in paragraphs]

                article_data = {
                    'link': link,
                    'title': title,
                    'subtitle': subtitle,
                    'paragraphs': main_paragraphs
                }
                data.append(article_data)

            except Exception as e:
                print(f"Error occurred while scraping {link}: {str(e)}")

        # Return the scraped data
        return ScrapingToolOutput(data=data)

In [19]:
scraping_tool = WebScrapingTool()

In [28]:
# Run the tool
result = scraping_tool._run(
    urls=[
        "https://www.mylondon.news/all-about/crime",
        "https://www.birminghammail.co.uk/all-about/crime",
        "https://www.manchestereveningnews.co.uk/all-about/crime",
        "https://www.liverpoolecho.co.uk/all-about/crime",
        "https://www.walesonline.co.uk/all-about/crime"
    ],
    num_links=7
)

In [30]:
# Save the result to JSON
with open('crew_scraped_data.json', 'w') as f:
    json.dump(result.dict(), f, indent=4)
    print("Data saved to crew_scraped_data.json")

Data saved to crew_scraped_data.json
